# Deep Learning - Notebook 20
## Computer Vision & Convolutional Neural Networks (CNNs) 

Welcome to Module 20.

In Notebook 19, we learned how fully connected Dense Neural Networks process tabular data. However, if you try to pass a high-resolution color photograph (e.g., 1024 x 1024 pixels x 3 color channels = 3,145,728 input numbers) into a Dense Neural Network, the model will require billions of weight parameters. It will run out of memory and immediately overfit.

Dense networks also destroy spatial relationships because they flatten image grids into 1D vectors.

To process visual data, we turn to **Convolutional Neural Networks (CNNs)**.

In this notebook, we master computer vision, spatial feature extraction, classic CNN architectures, transfer learning, object detection, and image segmentation using an anchor scenario:
* **Autonomous Vehicle Vision & Medical X-Ray Diagnostic Classifier:** Building visual AI models that analyze X-ray lung scans to detect pneumonia and process camera feeds for self-driving vehicles.

For every topic, we answer three questions:
* What is this in plain English?
* Why are we learning this?
* How is this used in Artificial Intelligence?

---
## Section 1: Images as Tensors & The Convolution Operation

### 1. What is this in plain English?
* **Images as Tensors:** To a computer, a color photograph is a 3D Tensor of numbers with shape `(Height, Width, Channels)`. RGB images have 3 color channels (Red, Green, Blue), where every pixel is a number from 0 to 255.
* **Convolution Layer:** Instead of connecting every pixel to every neuron, a CNN slides a small filter matrix (called a **Kernel**, e.g., 3x3 pixels) across the image. The kernel computes dot products at each position to detect specific visual features (like horizontal edges, vertical edges, or color gradients).
* **Feature Map:** The output matrix generated by sliding a kernel across an image. It highlights where specific visual features were detected.

### 2. Convolution Parameters:
* **Kernel Size:** Size of the sliding window (typically 3x3 or 5x5).
* **Stride:** How many pixels the filter moves at each step (Stride = 1 moves 1 pixel at a time; Stride = 2 skips every other pixel, cutting feature map size in half).
* **Padding:** Adding zero-value borders around the image (`padding='same'`) so the feature map maintains the exact same spatial dimensions as the original input image.

### 3. How is this used in AI?
* Extracting visual features in self-driving car cameras, facial recognition, and medical imaging.

In [1]:
# [Code Cell]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")

# 1. VISUALIZING AN IMAGE AS A 2D MATRIX (NUMPY)
# Creating a synthetic 8x8 greyscale image representing a vertical edge
synthetic_image = np.array([
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0],
    [0,  0,  0,  255, 255, 255, 0, 0]
], dtype=float)

# 2. DEFINING A VERTICAL EDGE DETECTION KERNEL (3x3)
vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

# 3. MANUAL 2D CONVOLUTION FUNCTION
def apply_2d_convolution(image, kernel):
    img_h, img_w = image.shape
    k_h, k_w = kernel.shape
    
    out_h = img_h - k_h + 1
    out_w = img_w - k_w + 1
    feature_map = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            # Extract image patch
            patch = image[i:i+k_h, j:j+k_w]
            # Compute element-wise multiplication and sum (Dot product)
            feature_map[i, j] = np.sum(patch * kernel)
            
    return feature_map

feature_map = apply_2d_convolution(synthetic_image, vertical_edge_kernel)

print("Original Image Shape:", synthetic_image.shape)
print("Feature Map Shape (Valid Padding):", feature_map.shape)
print("\nGenerated Feature Map (Vertical Edge Detection):\n", feature_map.astype(int))

Original Image Shape: (8, 8)
Feature Map Shape (Valid Padding): (6, 6)

Generated Feature Map (Vertical Edge Detection):
 [[   0  765  765    0 -765 -765]
 [   0  765  765    0 -765 -765]
 [   0  765  765    0 -765 -765]
 [   0  765  765    0 -765 -765]
 [   0  765  765    0 -765 -765]
 [   0  765  765    0 -765 -765]]



---
## Section 2: Pooling Layers & Spatial Invariance

### 1. What is this in plain English?
As a CNN extracts features, feature maps become deep and heavy. **Pooling Layers** act as downsamplers. 

A **Max Pooling Layer** slides a small window (typically 2x2) across a feature map and keeps ONLY the single largest value in each window.

### 2. Why are we learning this?
* **Dimensionality Reduction:** A 2x2 Max Pooling layer with Stride 2 cuts the spatial height and width of feature maps by 50% (reducing memory usage by 75%!).
* **Translation Invariance:** If a medical scan shows a tumor shifted 2 pixels to the left, Max Pooling ensures the model still detects the feature regardless of its exact pixel coordinate.

### 3. How is this used in AI?
* Compressing visual information between convolutional layers in deep visual networks.

In [3]:
def apply_max_pooling(feature_map, pool_size=2, stride=2):
    f_h, f_w = feature_map.shape
    out_h = (f_h - pool_size) // stride + 1
    out_w = (f_w - pool_size) // stride + 1
    
    pooled_map = np.zeros((out_h, out_w))
    
    for i in range(0, out_h):
        for j in range(out_w):
            r_start = i * stride
            c_start = j * stride
            patch = feature_map[r_start:r_start+pool_size, c_start:c_start+pool_size]
            pooled_map[i, j] = np.max(patch)
            
    return pooled_map

sample_feature_map = np.array([
    [10, 20, 0,  5],
    [40, 80, 12, 3],
    [5,  15, 90, 2],
    [8,  3,  1,  100]
])

pooled_output = apply_max_pooling(sample_feature_map, pool_size=2, stride=2)

print("Sample Feature Map (4x4):\n", sample_feature_map)
print("\nAfter 2x2 Max Pooling (2x2 Output):\n", pooled_output)

Sample Feature Map (4x4):
 [[ 10  20   0   5]
 [ 40  80  12   3]
 [  5  15  90   2]
 [  8   3   1 100]]

After 2x2 Max Pooling (2x2 Output):
 [[ 80.  12.]
 [ 15. 100.]]


---
## Section 3: Classic CNN Architectures (Evolution of Computer Vision)

### 1. What is this in plain English?
Over the past 25 years, AI researchers designed specific structural blueprints for stacking Convolutional, Pooling, and Dense layers. Understanding these classic architectures allows you to choose the right model for your task.

### 2. The Vision Architecture Timeline:
1. **LeNet-5 (1998):** Created by Yann LeCun for handwritten digit recognition (ZIP codes). Used Conv -> Pool -> Conv -> Pool -> Dense.
2. **AlexNet (2012):** Created by Alex Krizhevsky and Geoffrey Hinton. Won the ImageNet competition and launched the modern Deep Learning revolution by using GPUs, ReLU, and Dropout.
3. **VGG (VGG16 / VGG19 - 2014):** Simplified design by using small 3x3 convolution filters stacked deeply. Very effective, but computationally heavy.
4. **ResNet (Residual Networks - 2015):** Solved the "Degradation Problem" in deep networks using **Skip Connections (Residual Blocks)**:
   $$y = f(x) + x$$
   Skip connections pass the original input $x$ directly to deeper layers, allowing networks to train stably at 50, 101, or 152 layers deep!
5. **Inception (GoogLeNet - 2014):** Uses parallel convolution blocks ($1 \times 1, 3 \times 3, 5 \times 5$) inside a single layer to capture objects at multiple scales simultaneously.

In [5]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

    # CNN ARCHITECTURE FOR MEDICAL X-RAY CLASSIFICATION (128x128 RGB Images)
    model_cnn = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 2
        Conv2D(64, (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Block 3
        Conv2D(128, (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        
        # Dense Classification Head
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid") # Binary: Pneumonia vs Normal
    ])

    model_cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    
    print("--- Medical X-Ray CNN Architecture Summary ---")
    model_cnn.summary()

except ImportError:
    print("Note: 'tensorflow' package not installed. Keras CNN architecture demonstrated above.")

Note: 'tensorflow' package not installed. Keras CNN architecture demonstrated above.


---
## Section 4: Transfer Learning
### Standing on the Shoulders of Visual Giants

### 1. What is this in plain English?
Training a deep CNN from scratch requires millions of labeled photos and weeks of expensive GPU compute time.

**Transfer Learning** allows you to download a state-of-the-art model (like ResNet50 or VGG16) pre-trained on the ImageNet dataset (14 million images across 1,000 categories).

You **freeze** the early convolutional layers (which already know how to detect edges, curves, and shapes) and re-train ONLY the final classification layer on your custom dataset (e.g., medical X-rays or autonomous driving road signs).

### 2. Why are we learning this?
Transfer Learning allows you to achieve 95%+ accuracy on custom image classification datasets with as few as 200 training images in minutes!

### 3. How is this used in AI?
* Deployed in production computer vision systems at hospitals, robotics labs, and satellite imaging companies.

In [13]:
# TRANSFER LEARNING WITH PRE-TRAINED RESNET50

try:
    from tensorflow.keras.applications import ResNet50
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

    # Step 1: Load pre-trained ResNet50 without the final 1000-class ImageNet head
    base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(128, 128, 3))

    # Step 2: Freeze all convolutional base layers
    for layer in base_model.layers:
        layer.trainable = False

    # Step 3: Attach custom classification head for Medical Diagnostic X-Rays
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation="relu")(x)
    predictions = Dense(1, activation="sigmoid")(x) # Binary output

    # Step 4: Combine into final model
    transfer_model = Model(inputs=base_model.input, outputs=predictions)
    transfer_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    print("--- Transfer Learning Model Successfully Constructed ---")
    print(f"Total Base Layers (Frozen): {len(base_model.layers)}")
    print(f"Total Model Parameters:     {transfer_model.count_params():,}")

except ImportError:
    print("Note: 'tensorflow' package not installed. Transfer Learning syntax demonstrated above.")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 65s 1us/step
--- Transfer Learning Model Successfully Constructed ---
Total Base Layers (Frozen): 175
Total Model Parameters:     23,850,113


---
## Section 5: Advanced Computer Vision
### Object Detection & Image Segmentation

### 1. What is this in plain English?
* **Image Classification:** "Is there a car in this photo?" (Single label for the whole image).
* **Object Detection (YOLO, R-CNN):** "WHERE are the objects in this photo?" The model predicts bounding boxes $[x, y, \text{width}, \text{height}]$ and class labels around multiple objects simultaneously (e.g., Pedestrian at coordinate A, Car at coordinate B).
  * **YOLO (You Only Look Once):** Processes the entire image in a single forward pass at 60+ FPS, making it the industry standard for real-time video analytics.
  * **R-CNN / Faster R-CNN:** Uses region proposal networks for maximum precision.
* **Image Segmentation (U-Net, Mask R-CNN):** Classifies **every single individual pixel** in the image. Useful for autonomous driving (marking exact road pixels vs sidewalk pixels) and medical tumor boundary outline.

### 2. Intersection over Union (IoU):
To evaluate how accurately an object detection model placed a bounding box, we calculate **Intersection over Union (IoU)**:
$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$
An $\text{IoU} \ge 0.50$ is considered a successful detection.

In [14]:
# COMPUTING INTERSECTION OVER UNION (IoU) FOR OBJECT DETECTION

def calculate_iou(box_a, box_b):
    """
    box_a and box_b format: [x1, y1, x2, y2]
    (x1, y1) = Top-Left corner, (x2, y2) = Bottom-Right corner
    """
    # Determine intersection coordinates
    x_left = max(box_a[0], box_b[0])
    y_top = max(box_a[1], box_b[1])
    x_right = min(box_a[2], box_b[2])
    y_bottom = min(box_a[3], box_b[3])
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0 # No overlap
        
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    
    box_a_area = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    box_b_area = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    
    union_area = box_a_area + box_b_area - intersection_area
    
    return intersection_area / union_area

# Ground Truth Bounding Box around a pedestrian: [x1, y1, x2, y2]
ground_truth_box = [50, 50, 200, 200]

# AI Model Predicted Bounding Box
predicted_box    = [60, 50, 210, 200]

iou_score = calculate_iou(ground_truth_box, predicted_box)

print("Ground Truth Box: ", ground_truth_box)
print("Predicted Box:    ", predicted_box)
print(f"Bounding Box IoU Score: {iou_score:.4f}")
print("Detection Status: ", "SUCCESS (IoU >= 0.5)" if iou_score >= 0.5 else "FAILED")

Ground Truth Box:  [50, 50, 200, 200]
Predicted Box:     [60, 50, 210, 200]
Bounding Box IoU Score: 0.8750
Detection Status:  SUCCESS (IoU >= 0.5)


---
## Hands-on Workshop: Computer Vision Mechanics

Complete these four practical challenges to build your computer vision capabilities.

### Challenge 1: Horizontal Edge Filter (Junior Level)

**Scenario:** Compute a 2D convolution using a horizontal edge detection kernel.

**Task Requirements:**
1. Given a 4x4 image patch `img = np.array([[10, 10, 10, 10], [10, 10, 10, 10], [0, 0, 0, 0], [0, 0, 0, 0]])`.
2. Define a 3x3 horizontal edge kernel `kernel = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])`.
3. Compute 2D convolution for valid padding (producing a 2x2 output feature map).
4. Print the resulting feature map.

In [4]:
# --- CHALLENGE 1: YOUR WORKSPACE ---




### Challenge 2: Max Pooling Dimensionality Reduction (Mid Level)

**Scenario:** Reduce feature map dimensions using 2x2 Max Pooling with Stride 2.

**Task Requirements:**
1. Given a 4x4 feature map `fm = np.array([[5, 12, 3, 8], [9, 2, 14, 1], [1, 7, 6, 11], [4, 0, 2, 15]])`.
2. Apply 2x2 Max Pooling with Stride 2.
3. Print the pooled 2x2 matrix.

In [3]:
# --- CHALLENGE 2: YOUR WORKSPACE ---




### Challenge 3: PyTorch Conv2D Layer Construction (Senior Level)

**Scenario:** Construct a PyTorch Conv2D layer and check tensor shapes.

**Task Requirements:**
1. Import `torch` and `torch.nn as nn`.
2. Create a dummy image batch tensor `x = torch.randn(1, 3, 64, 64)` (Batch=1, Channels=3, Height=64, Width=64).
3. Initialize `conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)`.
4. Pass `x` through `conv` and print `out.shape` (Expected: `torch.Size([1, 16, 64, 64])`).

In [2]:
# --- CHALLENGE 3: YOUR WORKSPACE ---




### Challenge 4: Bounding Box Intersection over Union (Architect Level)

**Scenario:** Compute IoU for an autonomous vehicle obstacle detection system.

**Task Requirements:**
1. Ground Truth Box: `gt = [100, 100, 300, 300]` (Area = 200x200 = 40,000).
2. Predicted Box: `pred = [150, 100, 350, 300]` (Area = 200x200 = 40,000).
3. Compute Intersection Area (Overlap between X=150 to 300, Y=100 to 300 -> 150x200 = 30,000).
4. Compute Union Area ($40,000 + 40,000 - 30,000 = 50,000$).
5. Compute IoU score ($30,000 / 50,000 = 0.60$) and print.

In [1]:
# --- CHALLENGE 4: YOUR WORKSPACE ---


